# Evaluating large language models

The last years have seen an explosion of freely available large language models. But how to choose which is the right one for us / our task? Let's have a look at some metrics and methods for evaluating generative language models.

## ML Evaluation Techniques

If there is a first rule of machine learning, it is the following: _Do not evaluate your model on the same data that was used to train it_
 
Why? Because this is a recipe for selecting an **overfitted model**: One that performs well on the training data but generalizes poorly and is therefore likely to fail in practice. Imagine a classifier that has just memorized the labels for the training set. It will perform perfectly if it sees the training set again, but has learned nothing that will help it to classify new samples.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/overfitting.png" style="max-width: 500px; max-height: 400px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Overfitting example: The green boundary overfits the training data, while the black boundary generalizes better — by Chabacano, <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>, via <a href="https://commons.wikimedia.org/wiki/File:Overfitting.svg">Wikimedia Commons</a>
  </div>
</div>

### Train-test split

So what can we do instead? The most straightforward answer is to use a train-test-split for our ML workflow:

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/train_test_split.png" style="max-width: 650px; max-height: 400px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Train-test split — by EpochFail, <a href="https://creativecommons.org/licenses/by-sa/4.0/">CC BY-SA 4.0</a>, via <a href="https://commons.wikimedia.org/wiki/File:Machine_learning_nutshell_--_Split_into_train-test_set.svg">Wikimedia Commons</a>
  </div>
</div>

1. Split the data randomly into two sets. (A typical ratio is 80/20, assuming that training needs more examples than testing.)
2. Train the model on the first set.
3. Evaluate the models's performance on the second set.

### But what if it is not our own model?

Most of us likely don't have the resources to train our own LLM, and so will use a model someone else has trained and released. Problem is, we often don't know on what data the model was trained on, which makes it impossible to uphold our train-test-split. 

There is a range of commonly-used benchmark datasets for LLMs (which we'll look at soon). The standards of good science dictate that models should not be trained on these datasets. But there is no way for us to know for sure. Therefore it is important to stay vigilant: if a model is performing surprisingly well on some benchmark, always consider that it might have been trained on it.

## Classic NLP evaluation metrics

### Word error rate

[**Word Error Rate (WER)**](https://en.wikipedia.org/wiki/Word_error_rate) is typically used for evaluating speech transcription and text translation model. It is calculated by counting the number of added/deleted/substituted words compared to the ground truth.
$$\text{WER} = \frac{\text{S} + \text{D} + \text{I}}{\text{N}}$$


where:
- **S** is the number of substitutions (words that are incorrectly recognized),
- **D** is the number of deletions (words that are missed),
- **I** is the number of insertions (extra words that are added),
- **N** is the total number of words in the reference text.


For example, let's assume our speech transcription model has produced:

In [ ]:
truth = ["The", "eagle", "has", "landed"]
predicted = ["The", "igel", "landed"]

In [ ]:
# we can count two errors:
# 1x substitution (eagle -> igel)
# 1x missing (has)  
num_errors = 2

In [ ]:
# we then divide the number of errors by the length of the ground truth
word_error_rate = num_errors / len(truth)
word_error_rate

The word error rate works well for speech transcription models, were there is only one correct answer. For translation, where multiple possible answers can be correct, it is best if the predicted sentence is compared against multiple ground truth translations, done by different human translators.

### BLEU

BLEU is another text translation metric that is quite similar to Word error rate. It counts the number of words/n-grams that are the same in ground truth and prediction. 

In [ ]:
# we calculate the number of words (or n-grams) that are the same in truth and prediction
overlap = set(truth) & set(predicted)
overlap

In [ ]:
# and divide it by the length of the prediction
bleu = len(overlap) / len(predicted)
bleu

In [ ]:
# in order to avoid the model "cheating" by outputting very short phrases, a "brevity penalty" is added
penalty = 0.71 # a bit more complicated to calculate, not important here

bleu = bleu * penalty
bleu

Like with the Word error rate, it is recommended to compare a predicted sentence against multiple ground truth translations. 

Despite its simplicity, BLEU has been show to correlate highly with human judgments of quality, when calculated over a large machine translation corpus. 

### Perplexity

Perplexity is a metric for evaluating generative text models. It is a measure of how __"surprised"__ a model is by a sentence.

Remember what we train our generative models to do: given some previous words, we want it to predict the next word. To be precise, we want it to tell us, out of all possible next words, which are the most probable ones.

##### Calculation process:

1. Feed test sentence to our model, word by word.
2. At each word, store probability that the model assigns to that word
3. Average the probabilities
4. $\textit{perplexity} = \frac{1}{\textit{overall probability}}$

This means that the higher the perplexity of a test sentence, the more "surprised" the model is by that sentence. If our model works well, we should see:

* "The lion ate the rabbit" -> low perplexity.
* "The rabbit ate the lion" -> higher perplexity
* "Ate rabbit the the lion" -> very high perplecity

The advantage of perplexity is that it is not fixed to specific words, like Word error rate and BLEU. However, it does not really measure text content, whether what the model generated is what we wanted.

### Helpful libraries

The "evaluate" library from hugging face implements all the 3 metrics from above and many many more.

In [ ]:
from ai_dojo import show

In [ ]:
show.github_repo("https://github.com/huggingface/evaluate")

In [ ]:
import evaluate

In [ ]:
wer = evaluate.load("wer")
bleu = evaluate.load("bleu")

In [ ]:
predicted_str = " ".join(predicted)
truth_str = " ".join(truth)

In [ ]:
wer.compute(
    predictions=[predicted_str], 
    references=[truth_str]
)

In [ ]:
# numbers differ because the calculation is a bit more complicated than what we showed above
bleu.compute(
    predictions=[predicted_str],
    references=[truth_str],
)

## LLM benchmarks

There are a number of benchmarks for measuring the reasoning capabilities of LLMs. Let's have a look at three common ones:

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/screenshot-leaderboard.png" style="max-width: 900px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Screenshot from <a href="https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard">Hugging Face Open LLM Leaderboard</a>
  </div>
</div>


### AI2 Reasoning Challenge (ARC)

ARC is a dataset released in 2018 by the Allen Institute for AI. It can be downloaded at https://allenai.org/data/arc

ARC consists of ~8k multiple-choice science questions.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/metrics_arc.png" style="max-width: 700px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Reasoning types in the AI2 Reasoning Challenge (ARC) with example questions — Table 5 from Clark et al., <a href="https://arxiv.org/abs/1803.05457">"Think you have Solved Question Answering? Try ARC, the AI2 Reasoning Challenge"</a> (2018)
  </div>
</div>

### Measuring massive multitask language understanding (MMLU)

In [ ]:
show.github_repo("https://github.com/hendrycks/test")

This dataset is a set of multiple-choice questions spanning subjects in the humanities, social sciences, hard sciences, medicine, etc. In total there are ~16000 questions.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/metrics_mmlu_1.png" style="max-width: 700px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    MMLU example questions — Figure 4 from Hendrycks et al., <a href="https://arxiv.org/abs/2009.03300">"Measuring Massive Multitask Language Understanding"</a> (2021)
  </div>
</div>

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/metrics_mmlu_2.png" style="max-width: 700px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    MMLU example question — Figure 2 from Hendrycks et al., <a href="https://arxiv.org/abs/2009.03300">"Measuring Massive Multitask Language Understanding"</a> (2021)
  </div>
</div>

### Grade School Math (GSM8K)

In [ ]:
show.github_repo("https://github.com/openai/grade-school-math")

This dataset consists of ~8k elementary school text-based math questions.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/metrics_gsm8k.png" style="max-width: 700px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    GSM8K (Grade School Math) example problems — Figure 1 from Cobbe et al., <a href="https://arxiv.org/abs/2110.14168">"Training Verifiers to Solve Math Word Problems"</a> (2021)
  </div>
</div>

### Helpful libraries

In [ ]:
show.github_repo("https://github.com/confident-ai/deepeval")

```
from deepeval.benchmarks.mmlu import MMLU
...

benchmark = MMLU()
results = benchmark.evaluate(model=mistral_7b, batch_size=5)
print("Overall Score: ", results)
```

### Thoughts on those benchmarks

You have perhaps noted that these benchmarks have one thing in common: __the answer is always simple to check__, it is either right or wrong. 

This is in contrast to the NLP metrics we discussed earlier, where we always looked at the text directly and tried to put some number on how good or bad it is. Which is, of course, much harder to do.

## LLM-as-a-Judge 

With LLM-as-a-Judge, you use another (preferably more capable) LLM to judge your own LLM.

Basic idea:
1. Your LLM is asked to do some complex task
2. The Judge LLM then judges how good the answer from your LLM is.

Advantages:
* Quick and cheap
* Can easily be target to your task at hand

Disadvantages:
* You are dependent on the Judge LLM, which you may not know much about
  * e.g. GPT4 as an judge seems to prefer GPT4-based models 
  * judges seem to prefer longer answers

LLM-as-a-Judge has been getting some traction recently and different LLM-as-a-Judge leaderboards have been proposed:

In [ ]:
show.github_repo("https://github.com/tatsu-lab/alpaca_eval")

The FastChat repo contains an [LLM judge and leaderboard](https://github.com/lm-sys/FastChat/tree/main/fastchat/llm_judge)

In [ ]:
show.github_repo("https://github.com/lm-sys/FastChat")

## System evaluation

If we are building a chat-bot that helps customers choose resources they need, then we don't care much if this model can also do elementary school math. We do care, that the model can give reasonable and helpful advice and that customers are happy and feel well advised.

This brings us to a 3-tier evaluation strategy:

1. Choose candidate model(s) using the general benchmarks listed above, in particular benchmarks that are similar to your task.
2. Create your own test dataset specific to the task you are solving. For example, different potential user requests together with what you would expect the model to answer. Make sure that it is easily measurable, e.g. using the multiple choice format. 
3. Measure how the end users responds to the system, e.g. customer satisfaction. 


Ideally, the results from (2) and (3) correlate, i.e. better model -> happier user, but it is usually not that simple. There are many additional bits and pieces that go into building an AI system beyond just the model.



If you are working without (2), you are flying blind. Ad-hoc looking at a few chat messages and giving the thumbs-up or thumbs-down is not an advisable evaluation strategy. Good thing is: you can start small with this, with just a few samples and grow your evaluation dataset over time, as you learn more about how users interact with the system.

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_